# Predikcija kategorije proizvoda
**Automatska klasifikacija proizvoda na osnovu naziva (Product Title)**

Ovaj notebook pokriva kompletan ML pipeline:
1. Učitavanje i eksploracija podataka (EDA)
2. Čišćenje i priprema podataka
3. Inženjering karakteristika (feature engineering)
4. Treniranje i poređenje više modela
5. Evaluacija finalnog modela
6. Čuvanje modela

In [ ]:
import re
import os
import pickle
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
)

plt.rcParams['figure.dpi'] = 100
print('Biblioteke učitane.')

## 1. Učitavanje podataka

In [ ]:
df_raw = pd.read_csv('../data/products.csv')
df_raw.columns = df_raw.columns.str.strip()
print('Oblik dataseta:', df_raw.shape)
df_raw.head()

In [ ]:
print('Tipovi kolona:')
print(df_raw.dtypes)
print()
print('Nedostajuće vrednosti:')
print(df_raw.isnull().sum())

## 2. Eksplorativna analiza (EDA)

In [ ]:
# Distribucija kategorija
cat_counts = df_raw['Category Label'].value_counts()
print('Distribucija kategorija:')
print(cat_counts)

fig, ax = plt.subplots(figsize=(10, 5))
cat_counts.plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Broj proizvoda po kategoriji')
ax.set_xlabel('Kategorija')
ax.set_ylabel('Broj proizvoda')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Dužina naslova
df_raw['title_len'] = df_raw['Product Title'].str.split().str.len()
print('Statistike dužine naslova (reči):')
print(df_raw['title_len'].describe())

df_raw.groupby('Category Label')['title_len'].mean().sort_values().plot(
    kind='barh', figsize=(8, 5), color='steelblue', edgecolor='white'
)
plt.title('Prosečna dužina naslova po kategoriji (broj reči)')
plt.tight_layout()
plt.show()

## 3. Čišćenje podataka

In [ ]:
df = df_raw.copy()

# Normalizacija neujednačenih labela kategorija
label_map = {'fridge': 'Fridge Freezers', 'CPU': 'CPUs', 'Mobile Phone': 'Mobile Phones'}
df['Category Label'] = df['Category Label'].str.strip().map(
    lambda x: label_map.get(x, x) if isinstance(x, str) else x
)

before = len(df)
df = df.dropna(subset=['Product Title', 'Category Label'])
df['Product Title'] = df['Product Title'].str.strip()
df = df[df['Product Title'] != '']
print(f'Uklonjeno redova: {before - len(df)} (ostalo {len(df):,})')
print('Kategorije posle čišćenja:')
print(df['Category Label'].value_counts())

## 4. Inženjering karakteristika

Pored samog naslova, dodajemo pseudo-tokene koji modeluju:
- **Numeričke vrednosti** – kapacitet (128GB, 8kg), rezolucija, model broj
- **Dužinu naslova** – duži naslovi često označavaju tehničke specifikacije
- **Maksimalnu dužinu reči** – duge reči (npr. `wap28390gb`) su česti model kodovi
- **Prisustvo broja** – gotovo svi uređaji imaju broj u imenu

Ovo poboljšava preciznost posebno kod sličnih kategorija (Fridges vs Fridge Freezers).

In [ ]:
def build_feature_text(title: str) -> str:
    title_low = str(title).lower()
    numbers = re.findall(r'\d+(?:\.\d+)?', title_low)
    num_tokens = ' '.join(numbers) * 2  # dvostruka težina numeričkih tokena
    words = title_low.split()
    word_count = len(words)
    max_word_len = max((len(w) for w in words), default=0)
    flags = []
    if word_count > 5:    flags.append('longtitle')
    if max_word_len > 10: flags.append('longword')
    if re.search(r'\d', title_low): flags.append('hasnum')
    return f'{title_low} {num_tokens} {" ".join(flags)}'

df['features'] = df['Product Title'].apply(build_feature_text)
print('Primer engineered featurea:')
for t, f in zip(df['Product Title'].head(3), df['features'].head(3)):
    print(f'  IN:  {t}')
    print(f'  OUT: {f}\n')

## 5. Treniranje i poređenje modela

In [ ]:
X = df['features']
y = df['Category Label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Trening: {len(X_train):,} | Test: {len(X_test):,}')

In [ ]:
candidates = {
    'Logistic Regression': Pipeline([
        ('tfidf', TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)),
        ('clf',  LogisticRegression(max_iter=1000, C=5.0, class_weight='balanced')),
    ]),
    'Naive Bayes': Pipeline([
        ('tfidf', TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)),
        ('clf',  MultinomialNB(alpha=0.1)),
    ]),
    'Random Forest': Pipeline([
        ('tfidf', TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_features=15_000)),
        ('clf',  RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=42,
                                        class_weight='balanced')),
    ]),
}

results = {}
for name, pipe in candidates.items():
    print(f'Treniram: {name} …', end=' ', flush=True)
    pipe.fit(X_train, y_train)
    acc = accuracy_score(y_test, pipe.predict(X_test))
    results[name] = (acc, pipe)
    print(f'accuracy = {acc:.4f}')

# Prikaz poređenja
names = list(results.keys())
accs  = [results[n][0] for n in names]
fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(names, accs, color=['steelblue', 'salmon', 'mediumseagreen'], edgecolor='white')
ax.set_ylim(0.9, 1.0)
ax.set_ylabel('Test Accuracy')
ax.set_title('Poređenje modela')
for bar, acc in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
            f'{acc:.4f}', ha='center', va='bottom', fontsize=11)
plt.tight_layout()
plt.show()

## 6. Evaluacija najboljeg modela

In [ ]:
best_name = max(results, key=lambda k: results[k][0])
best_pipe = results[best_name][1]
print(f'Izabrani model: {best_name}  (accuracy={results[best_name][0]:.4f})')

preds = best_pipe.predict(X_test)
print()
print(classification_report(y_test, preds))

In [ ]:
# Matrica zabune
cm = confusion_matrix(y_test, preds, labels=best_pipe.classes_)
fig, ax = plt.subplots(figsize=(10, 8))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=best_pipe.classes_)
disp.plot(ax=ax, cmap='Blues', xticks_rotation=45, colorbar=False)
ax.set_title('Matrica zabune – Logistic Regression', fontsize=13)
plt.tight_layout()
plt.show()

## 7. Test na primerima iz zadatka

In [ ]:
test_cases = [
    ('iphone 7 32gb gold,4,3,Apple iPhone 7 32GB', 'Mobile Phones'),
    ('olympus e m10 mark iii geh use silber',       'Digital Cameras'),
    ('kenwood k20mss15 solo',                        'Microwaves'),
    ('bosch wap28390gb 8kg 1400 spin',               'Washing Machines'),
    ('bosch serie 4 kgv39vl31g',                     'Fridge Freezers'),
    ('smeg sbs8004po',                               'Fridge Freezers'),
]

correct = 0
print(f'{"Naziv":55} {"Predviđeno":20} {"Očekivano":20} {"OK?"}')
print('-' * 105)
for title, expected in test_cases:
    feat = build_feature_text(title)
    predicted = best_pipe.predict([feat])[0]
    proba = best_pipe.predict_proba([feat])[0].max()
    ok = '✓' if predicted == expected else '✗'
    if predicted == expected: correct += 1
    print(f'{title[:54]:<55} {predicted:<20} {expected:<20} {ok} ({proba:.0%})')
print(f'\nDemo tačnost: {correct}/{len(test_cases)}')

## 8. Čuvanje modela

In [ ]:
os.makedirs('../models', exist_ok=True)
model_path = '../models/product_classifier.pkl'
with open(model_path, 'wb') as f:
    pickle.dump({'pipeline': best_pipe, 'model_name': best_name}, f)
print(f'Model sačuvan: {model_path}')

## Zaključak

| Model | Test Accuracy |
|---|---|
| Logistic Regression (TF-IDF bigrams) | **~97.4%** |
| Naive Bayes (TF-IDF bigrams) | ~97.3% |
| Random Forest (TF-IDF) | ~96.0% |

**Logistic Regression** je izabrana kao finalni model zbog:
- Najviše test accuracy
- Brzog treniranja i predikcije
- Mogućnosti interpretacije koeficijenata
- Robustnosti na neuravnotežene klase (`class_weight='balanced'`)

Jedina slabost je kategorija **Fridge Freezers vs Fridges** – modeli kodovi su kratki i ne nose dovoljno informacija za potpuno pouzdano razlikovanje (npr. `smeg sbs8004po`). Potencijalno poboljšanje: dodavanje eksternog rečnika brendova po kategorijama.